## 查看data/*.parquet文件

In [1]:
# Jupyter Notebook 单元格代码
import os
from pathlib import Path
import pandas as pd
import numpy as np
from datetime import datetime
from io import StringIO

PROJECT_ROOT = Path.cwd().parent.resolve()

# ========== 全局变量定义（请根据需求修改）==========
# 数据目录路径
DATA_DIR = str(PROJECT_ROOT / "datasets/Packing_Box/auto/Packing_Box_episode_1397/meta/episodes/chunk-000")

# 是否保存为txt文件（True: 保存, False: 不保存）
SAVE_TO_TXT = True

# txt文件输出目录（设置为当前工作目录）
OUTPUT_DIR = os.getcwd()  # 当前命令运行的目录

# txt文件名前缀（会自动添加时间戳）
TXT_PREFIX = "dataset_analysis"
# =================================================

# 执行分析
target_dir = Path(DATA_DIR)

# 用于存储输出内容
output_lines = []

def print_and_log(text):
    """同时打印到控制台和日志"""
    print(text)
    if SAVE_TO_TXT:
        output_lines.append(text)

def safe_nunique(series):
    """安全计算唯一值数量，处理numpy数组等不可哈希类型"""
    try:
        return series.nunique()
    except TypeError:
        # 对于包含不可哈希类型的列，尝试转换为字符串后再计算
        try:
            return series.astype(str).nunique()
        except:
            return len(series.drop_duplicates())

def get_column_info(col_name, series):
    """获取列的详细信息"""
    info_lines = []
    info_lines.append(f"\n列名: {col_name}")
    info_lines.append(f"  - 数据类型: {series.dtype}")
    info_lines.append(f"  - 非空值数量: {series.count():,} / {len(series):,}")
    info_lines.append(f"  - 空值数量: {series.isnull().sum():,}")
    
    # 对于数值类型显示统计信息
    if pd.api.types.is_numeric_dtype(series):
        non_null = series.dropna()
        if len(non_null) > 0:
            info_lines.append(f"  - 最小值: {non_null.min()}")
            info_lines.append(f"  - 最大值: {non_null.max()}")
            info_lines.append(f"  - 平均值: {non_null.mean():.4f}")
            info_lines.append(f"  - 标准差: {non_null.std():.4f}")
        else:
            info_lines.append(f"  - 统计信息: 全为空值")
    
    # 对于对象类型（字符串、列表等）显示示例
    elif series.dtype == 'object':
        # 获取第一个非空值
        non_null = series.dropna()
        if len(non_null) > 0:
            first_valid = non_null.iloc[0]
            info_lines.append(f"  - 数据类型示例: {type(first_valid).__name__}")
            
            # 如果是列表或数组，显示长度和维度信息
            if isinstance(first_valid, (list, tuple, np.ndarray)):
                info_lines.append(f"  - 序列长度: {len(first_valid)}")
                if len(first_valid) > 0:
                    info_lines.append(f"  - 第一个元素类型: {type(first_valid[0]).__name__}")
                    if isinstance(first_valid, np.ndarray):
                        info_lines.append(f"  - 数组形状: {first_valid.shape}")
                    # 显示示例值
                    sample_values = first_valid[:3] if len(first_valid) >= 3 else first_valid
                    info_lines.append(f"  - 示例(前3个): {sample_values}")
            else:
                # 字符串或其他类型
                sample_str = str(first_valid)[:100]
                if len(str(first_valid)) > 100:
                    sample_str += "..."
                info_lines.append(f"  - 示例值: {sample_str}")
            
            # 显示唯一值数量（安全计算）
            unique_count = safe_nunique(series)
            info_lines.append(f"  - 唯一值数量: {unique_count:,}")
            if unique_count <= 10 and unique_count > 0:
                try:
                    unique_vals = series.unique()
                    # 转换不可哈希类型为字符串显示
                    unique_vals_str = [str(v)[:50] for v in unique_vals]
                    info_lines.append(f"  - 唯一值: {unique_vals_str}")
                except:
                    info_lines.append(f"  - 唯一值: 无法显示（包含不可哈希类型）")
        else:
            info_lines.append(f"  - 所有值都为空")
    
    # 对于其他类型
    else:
        if len(series) > 0:
            info_lines.append(f"  - 示例值: {series.iloc[0]}")
    
    return info_lines

# 开始分析
print_and_log("=" * 100)
print_and_log(f"数据集分析报告")
print_and_log(f"生成时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print_and_log("=" * 100)
print_and_log(f"目标目录: {target_dir}")
print_and_log(f"目录是否存在: {target_dir.exists()}")
print_and_log(f"当前工作目录: {os.getcwd()}")
print_and_log("=" * 100)

if not target_dir.exists():
    raise FileNotFoundError(f"目录不存在: {target_dir}")

# 递归查找所有 .parquet 文件
parquet_files = sorted(target_dir.rglob("*.parquet"))

print_and_log(f"\n共找到 {len(parquet_files)} 个 parquet 文件")

if len(parquet_files) == 0:
    raise FileNotFoundError(f"目录中没有找到任何 .parquet 文件: {target_dir}")

# 获取第一个文件
first_file = parquet_files[0]
print_and_log(f"\n第一个文件路径: {first_file}")
print_and_log(f"文件名: {first_file.name}")
print_and_log(f"相对路径: {first_file.relative_to(target_dir) if first_file.parent != target_dir else first_file.name}")

# 读取parquet文件
print_and_log("\n正在读取数据...")
df = pd.read_parquet(first_file)

# 文件基本信息
file_size_mb = round(first_file.stat().st_size / 1024 / 1024, 2)
print_and_log("\n" + "=" * 100)
print_and_log("文件基本信息")
print_and_log("=" * 100)
print_and_log(f"文件大小: {file_size_mb} MB")
print_and_log(f"数据行数: {len(df):,}")
print_and_log(f"数据列数: {len(df.columns)}")
print_and_log(f"列名列表: {list(df.columns)}")

# 显示前几行数据（处理复杂数据类型）
print_and_log("\n" + "=" * 100)
print_and_log("前2行数据预览（简化显示）")
print_and_log("=" * 100)

for idx in range(min(2, len(df))):
    print_and_log(f"\n第 {idx} 行:")
    for col in df.columns:
        value = df[col].iloc[idx]
        if isinstance(value, np.ndarray):
            print_and_log(f"  {col}: array(shape={value.shape}, dtype={value.dtype})")
            print_and_log(f"      前5个值: {value[:5]}")
        elif isinstance(value, (list, tuple)):
            print_and_log(f"  {col}: {type(value).__name__}(len={len(value)})")
            print_and_log(f"      前5个值: {value[:5] if len(value) >= 5 else value}")
        else:
            print_and_log(f"  {col}: {value}")

# 显示基本信息
print_and_log("\n" + "=" * 100)
print_and_log("数据集基本信息")
print_and_log("=" * 100)
print_and_log(f"数据维度: {df.shape}")
print_and_log(f"列名及类型:")
for col in df.columns:
    print_and_log(f"  - {col}: {df[col].dtype}")

# 每列详细信息
print_and_log("\n" + "=" * 100)
print_and_log("每列详细信息")
print_and_log("=" * 100)

for col in df.columns:
    info_lines = get_column_info(col, df[col])
    for line in info_lines:
        print_and_log(line)

# 数据集整体描述（仅数值列）
print_and_log("\n" + "=" * 100)
print_and_log("数据集描述统计（仅数值列）")
print_and_log("=" * 100)
numeric_cols = df.select_dtypes(include=[np.number]).columns
if len(numeric_cols) > 0:
    print_and_log(df[numeric_cols].describe().to_string())
else:
    print_and_log("没有数值类型的列")

# 内存使用情况
print_and_log("\n" + "=" * 100)
print_and_log("内存使用情况")
print_and_log("=" * 100)
memory_usage = df.memory_usage(deep=True)
print_and_log(f"总内存使用: {memory_usage.sum() / 1024 / 1024:.2f} MB")
print_and_log("\n各列内存使用:")
for col in df.columns:
    print_and_log(f"  - {col}: {memory_usage[col] / 1024 / 1024:.2f} MB")

# 检查常见的数据集字段
print_and_log("\n" + "=" * 100)
print_and_log("数据集结构分析")
print_and_log("=" * 100)

# 检查常见的机器人数据集字段
common_fields = ['episode_index', 'frame_index', 'timestamp', 'action', 'observation.state', 
                 'observation.environment_state', 'task_index', 'index']
for field in common_fields:
    if field in df.columns:
        print_and_log(f"✓ 包含字段: {field}")
        
        # 显示字段详细信息
        if len(df) > 0:
            sample = df[field].iloc[0]
            if isinstance(sample, (list, tuple, np.ndarray)):
                print_and_log(f"  - {field} 类型: {type(sample).__name__}")
                if isinstance(sample, np.ndarray):
                    print_and_log(f"  - {field} 形状: {sample.shape}")
                print_and_log(f"  - {field} 长度: {len(sample)}")
                if len(sample) > 0:
                    print_and_log(f"  - {field} 数据类型: {type(sample[0]).__name__}")
            else:
                print_and_log(f"  - {field} 类型: {type(sample).__name__}")
                print_and_log(f"  - {field} 示例值: {sample}")

# 保存为txt文件
if SAVE_TO_TXT:
    # 使用当前工作目录
    output_dir = Path(OUTPUT_DIR)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # 生成文件名（使用数据目录名和数据文件名）
    data_dir_name = target_dir.parent.name + "_" + target_dir.name if target_dir.parent != target_dir else target_dir.name
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    txt_filename = f"{TXT_PREFIX}_{data_dir_name}_{first_file.stem}_{timestamp}.txt"
    txt_filepath = output_dir / txt_filename
    
    # 写入文件
    with open(txt_filepath, 'w', encoding='utf-8') as f:
        f.write('\n'.join(output_lines))
    
    print_and_log("\n" + "=" * 100)
    print_and_log(f"分析报告已保存至: {txt_filepath}")
    print_and_log(f"保存目录: {output_dir.absolute()}")
    print_and_log("=" * 100)
else:
    print_and_log("\n" + "=" * 100)
    print_and_log("未保存txt文件（SAVE_TO_TXT = False）")
    print_and_log("=" * 100)

print_and_log("\n分析完成！")

数据集分析报告
生成时间: 2026-05-23 22:17:23
目标目录: /data/SJJ/UBT/lerobot_0.5.1/datasets/Packing_Box/auto/Packing_Box_episode_1397/meta/episodes/chunk-000
目录是否存在: True
当前工作目录: /data/SJJ/UBT/lerobot_0.5.1/datasets

共找到 1 个 parquet 文件

第一个文件路径: /data/SJJ/UBT/lerobot_0.5.1/datasets/Packing_Box/auto/Packing_Box_episode_1397/meta/episodes/chunk-000/file-000.parquet
文件名: file-000.parquet
相对路径: file-000.parquet

正在读取数据...

文件基本信息
文件大小: 1.73 MB
数据行数: 1,397
数据列数: 135
列名列表: ['episode_index', 'tasks', 'length', 'data/chunk_index', 'data/file_index', 'dataset_from_index', 'dataset_to_index', 'videos/observation.images.head_left/chunk_index', 'videos/observation.images.head_left/file_index', 'videos/observation.images.head_left/from_timestamp', 'videos/observation.images.head_left/to_timestamp', 'videos/observation.images.head_right/chunk_index', 'videos/observation.images.head_right/file_index', 'videos/observation.images.head_right/from_timestamp', 'videos/observation.images.head_right/to_timestamp', 'videos

## 查看 episode_index 分布

读取合并后的 parquet 文件，统计 `episode_index` 列的分布情况（最大值、最小值、唯一值数量、每个 episode 包含的帧数）。

**使用示例：**
- 修改 `parquet_path` 为目标 parquet 目录路径（指向 `data/chunk-000` 目录）
- 运行后查看每个 episode 的帧数分布是否均匀

In [4]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.resolve()

parquet_path = PROJECT_ROOT / "datasets/Packing_Box/auto/Packing_Box_merged/data/chunk-000"

df = pd.read_parquet(parquet_path)

print("文件路径:", parquet_path)
print("总行数:", len(df))
print("总列数:", len(df.columns))

print("\n列名:")
print(df.columns.tolist())

print("\nepisode_index 最大值:")
print(df["episode_index"].max())

print("\nepisode_index 最小值:")
print(df["episode_index"].min())

print("\nepisode_index 唯一数量:")
print(df["episode_index"].nunique())

print("\nepisode_index 分布:")
display(df["episode_index"].value_counts().sort_index())

文件路径: /data/SJJ/UBT/lerobot_0.5.1/datasets/Packing_Box/auto/Packing_Box_merged/data/chunk-000
总行数: 575039
总列数: 7

列名:
['action', 'observation.state', 'timestamp', 'frame_index', 'episode_index', 'index', 'task_index']

episode_index 最大值:
1025

episode_index 最小值:
0

episode_index 唯一数量:
1026

episode_index 分布:


episode_index
0       563
1       563
2       563
3       563
4       563
       ... 
1021    563
1022    563
1023    563
1024    563
1025    236
Name: count, Length: 1026, dtype: int64

## 处理observation.environment_state数据

In [6]:
# Jupyter Notebook 单元格代码
import os
from pathlib import Path
import pandas as pd
import numpy as np
from datetime import datetime
import warnings

PROJECT_ROOT = Path.cwd().parent.resolve()
warnings.filterwarnings('ignore')

# ========== 全局变量定义（请根据需求修改）==========
# 数据目录路径
DATA_DIR = str(PROJECT_ROOT / "datasets/Part_Sorting/26_5_11_episode_1000_obj_4/data/chunk-000")

# 是否保存为txt文件（True: 保存处理日志, False: 不保存）
SAVE_LOG = False

# 是否创建备份（True: 备份原始文件, False: 不备份）
CREATE_BACKUP = False

# 备份目录后缀
BACKUP_SUFFIX = "_backup"

# 是否覆盖原文件（False: 保存到新文件）
OVERWRITE_ORIGINAL = True

# 输出目录（当OVERWRITE_ORIGINAL=False时使用）
OUTPUT_DIR = None
# =================================================

def concatenate_state_columns(df):
    """
    将observation.environment_state拼接到observation.state后面
    """
    # 复制DataFrame避免修改原始数据
    df_processed = df.copy()
    
    # 确认需要的列存在
    if 'observation.state' not in df_processed.columns:
        raise KeyError("DataFrame中缺少 'observation.state' 列")
    if 'observation.environment_state' not in df_processed.columns:
        raise KeyError("DataFrame中缺少 'observation.environment_state' 列")
    
    print(f"  处理前 - observation.state shape: {df_processed['observation.state'].iloc[0].shape if len(df_processed) > 0 else 'N/A'}")
    print(f"  处理前 - observation.environment_state shape: {df_processed['observation.environment_state'].iloc[0].shape if len(df_processed) > 0 else 'N/A'}")
    
    # 拼接两个数组
    def concatenate_arrays(row):
        state = row['observation.state']
        env_state = row['observation.environment_state']
        # 使用np.concatenate拼接数组
        return np.concatenate([state, env_state])
    
    # 应用拼接函数
    df_processed['observation.state'] = df_processed.apply(concatenate_arrays, axis=1)
    
    print(f"  处理后 - observation.state shape: {df_processed['observation.state'].iloc[0].shape if len(df_processed) > 0 else 'N/A'}")
    
    # 删除observation.environment_state列
    df_processed = df_processed.drop('observation.environment_state', axis=1)
    
    return df_processed

def process_single_file(file_path, create_backup=True, overwrite=True, output_dir=None):
    """
    处理单个parquet文件
    """
    file_info = {
        'path': file_path,
        'success': False,
        'original_rows': 0,
        'original_cols': 0,
        'processed_cols': 0,
        'error': None
    }
    
    try:
        print(f"\n处理文件: {file_path.name}")
        
        # 读取parquet文件
        df = pd.read_parquet(file_path)
        file_info['original_rows'] = len(df)
        file_info['original_cols'] = len(df.columns)
        
        print(f"  原始数据形状: {df.shape}")
        print(f"  原始列: {list(df.columns)}")
        
        # 检查是否需要处理
        if 'observation.environment_state' not in df.columns:
            print(f"  跳过: 文件中没有 'observation.environment_state' 列")
            file_info['success'] = True
            file_info['processed_cols'] = len(df.columns)
            return file_info
        
        # 处理数据
        df_processed = concatenate_state_columns(df)
        file_info['processed_cols'] = len(df_processed.columns)
        
        print(f"  处理后数据形状: {df_processed.shape}")
        print(f"  处理后列: {list(df_processed.columns)}")
        
        # 保存文件
        if overwrite:
            # 如果需要备份，先创建备份
            if create_backup:
                backup_path = file_path.with_suffix(f'.parquet{BACKUP_SUFFIX}')
                if not backup_path.exists():
                    df.to_parquet(backup_path)
                    print(f"  已创建备份: {backup_path.name}")
                else:
                    print(f"  备份文件已存在: {backup_path.name}")
            
            # 覆盖原文件
            df_processed.to_parquet(file_path, index=False)
            print(f"  已覆盖原文件: {file_path.name}")
        else:
            # 保存到新文件
            if output_dir is None:
                output_path = file_path.parent / f"{file_path.stem}_processed.parquet"
            else:
                output_path = Path(output_dir) / f"{file_path.stem}_processed.parquet"
                output_path.parent.mkdir(parents=True, exist_ok=True)
            
            df_processed.to_parquet(output_path, index=False)
            print(f"  已保存到新文件: {output_path}")
        
        file_info['success'] = True
        
    except Exception as e:
        file_info['error'] = str(e)
        print(f"  错误: {e}")
    
    return file_info

# 开始处理
print("=" * 100)
print("数据集处理 - 拼接 observation.environment_state 到 observation.state")
print(f"开始时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 100)

# 检查目录
target_dir = Path(DATA_DIR)
print(f"\n目标目录: {target_dir}")
print(f"目录是否存在: {target_dir.exists()}")

if not target_dir.exists():
    raise FileNotFoundError(f"目录不存在: {target_dir}")

# 查找所有parquet文件
parquet_files = sorted(target_dir.rglob("*.parquet"))
# 过滤掉备份文件
parquet_files = [f for f in parquet_files if BACKUP_SUFFIX not in f.suffixes]

print(f"\n找到 {len(parquet_files)} 个parquet文件（不包括备份文件）")

if len(parquet_files) == 0:
    raise FileNotFoundError(f"目录中没有找到任何 .parquet 文件: {target_dir}")

# 处理参数显示
print(f"\n处理配置:")
print(f"  - 创建备份: {CREATE_BACKUP}")
print(f"  - 覆盖原文件: {OVERWRITE_ORIGINAL}")
print(f"  - 备份后缀: {BACKUP_SUFFIX}")
if not OVERWRITE_ORIGINAL:
    print(f"  - 输出目录: {OUTPUT_DIR if OUTPUT_DIR else '原文件目录'}")

# 处理所有文件
print("\n" + "=" * 100)
print("开始处理文件...")
print("=" * 100)

results = []
for i, file_path in enumerate(parquet_files, 1):
    print(f"\n进度: [{i}/{len(parquet_files)}]")
    result = process_single_file(
        file_path, 
        create_backup=CREATE_BACKUP,
        overwrite=OVERWRITE_ORIGINAL,
        output_dir=OUTPUT_DIR
    )
    results.append(result)

# 显示处理结果汇总
print("\n" + "=" * 100)
print("处理结果汇总")
print("=" * 100)

success_count = sum(1 for r in results if r['success'])
error_count = len(results) - success_count

print(f"\n总文件数: {len(results)}")
print(f"成功处理: {success_count}")
print(f"失败: {error_count}")

if error_count > 0:
    print("\n失败文件列表:")
    for r in results:
        if not r['success']:
            print(f"  - {r['path'].name}: {r['error']}")

# 显示第一个文件的处理前后对比
print("\n" + "=" * 100)
print("处理示例（第一个文件）")
print("=" * 100)

if results and results[0]['success']:
    first_file = parquet_files[0]
    if OVERWRITE_ORIGINAL and CREATE_BACKUP:
        backup_file = first_file.with_suffix(f'.parquet{BACKUP_SUFFIX}')
        if backup_file.exists():
            df_original = pd.read_parquet(backup_file)
            df_processed = pd.read_parquet(first_file)
            
            print(f"\n原始文件: {first_file.name}")
            print(f"  - 形状: {df_original.shape}")
            print(f"  - 列: {list(df_original.columns)}")
            if 'observation.state' in df_original.columns and len(df_original) > 0:
                print(f"  - observation.state shape: {df_original['observation.state'].iloc[0].shape}")
                print(f"  - observation.state dtype: {df_original['observation.state'].iloc[0].dtype}")
            if 'observation.environment_state' in df_original.columns and len(df_original) > 0:
                print(f"  - observation.environment_state shape: {df_original['observation.environment_state'].iloc[0].shape}")
            
            print(f"\n处理后文件: {first_file.name}")
            print(f"  - 形状: {df_processed.shape}")
            print(f"  - 列: {list(df_processed.columns)}")
            if 'observation.state' in df_processed.columns and len(df_processed) > 0:
                print(f"  - observation.state shape: {df_processed['observation.state'].iloc[0].shape}")
                print(f"  - observation.state dtype: {df_processed['observation.state'].iloc[0].dtype}")

# 保存处理日志
if SAVE_LOG:
    log_lines = []
    log_lines.append("=" * 100)
    log_lines.append(f"数据处理日志")
    log_lines.append(f"处理时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    log_lines.append("=" * 100)
    log_lines.append(f"\n目标目录: {DATA_DIR}")
    log_lines.append(f"处理配置:")
    log_lines.append(f"  - 创建备份: {CREATE_BACKUP}")
    log_lines.append(f"  - 覆盖原文件: {OVERWRITE_ORIGINAL}")
    log_lines.append(f"\n处理结果:")
    log_lines.append(f"  - 总文件数: {len(results)}")
    log_lines.append(f"  - 成功处理: {success_count}")
    log_lines.append(f"  - 失败: {error_count}")
    
    if error_count > 0:
        log_lines.append(f"\n失败文件详情:")
        for r in results:
            if not r['success']:
                log_lines.append(f"  - {r['path'].name}: {r['error']}")
    
    log_lines.append(f"\n处理文件列表:")
    for r in results:
        status = "✓" if r['success'] else "✗"
        log_lines.append(f"  {status} {r['path'].name}")
        if r['success']:
            log_lines.append(f"      形状: {r['original_rows']}x{r['original_cols']} -> {r['original_rows']}x{r['processed_cols']}")
    
    # 保存日志文件
    log_dir = Path(os.getcwd())
    log_filename = f"dataset_processing_log_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"
    log_filepath = log_dir / log_filename
    
    with open(log_filepath, 'w', encoding='utf-8') as f:
        f.write('\n'.join(log_lines))
    
    print(f"\n处理日志已保存至: {log_filepath}")

print("\n" + "=" * 100)
print(f"处理完成时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 100)

数据集处理 - 拼接 observation.environment_state 到 observation.state
开始时间: 2026-05-12 17:59:16

目标目录: /data/SJJ/UBT/lerobot_0.5.1/datasets/Part_Sorting/26_5_11_episode_1000_obj_4/data/chunk-000
目录是否存在: True

找到 756 个parquet文件（不包括备份文件）

处理配置:
  - 创建备份: False
  - 覆盖原文件: True
  - 备份后缀: _backup

开始处理文件...

进度: [1/756]

处理文件: episode_000000.parquet
  原始数据形状: (768, 8)
  原始列: ['observation.state', 'action', 'observation.environment_state', 'timestamp', 'frame_index', 'episode_index', 'index', 'task_index']
  处理前 - observation.state shape: (20,)
  处理前 - observation.environment_state shape: (28,)
  处理后 - observation.state shape: (48,)
  处理后数据形状: (768, 7)
  处理后列: ['observation.state', 'action', 'timestamp', 'frame_index', 'episode_index', 'index', 'task_index']
  已覆盖原文件: episode_000000.parquet

进度: [2/756]

处理文件: episode_000001.parquet
  原始数据形状: (768, 8)
  原始列: ['observation.state', 'action', 'observation.environment_state', 'timestamp', 'frame_index', 'episode_index', 'index', 'task_index']
  处理前 - obs

## 删除备份文件

递归搜索指定目录下的备份文件（如 `_backup`, `.bak` 等），支持预览模式和确认删除模式，并可记录操作日志。

**使用示例：**
- 设置 `DATA_DIR` 为目标目录路径
- 先设置 `CONFIRM_DELETE = False` 预览会删除哪些文件
- 确认无误后设置 `CONFIRM_DELETE = True` 执行删除

In [2]:
# Jupyter Notebook 单元格代码
import os
from pathlib import Path
from datetime import datetime

PROJECT_ROOT = Path.cwd().parent.resolve()

# ========== 全局变量定义 ==========
# 数据目录路径
DATA_DIR = str(PROJECT_ROOT / "datasets/Part_Sorting/test/data/chunk-000")

# 备份文件标识（支持多种标识）
BACKUP_KEYWORDS = ["_backup", ".backup", "_bak", ".bak", "backup_", "back_up", "parquet_backup"]

# 是否确认删除（True: 执行删除, False: 仅预览不删除）
CONFIRM_DELETE = True  # 改为True才会真正删除

# 是否递归搜索子目录
RECURSIVE = True

# 是否显示详细信息
VERBOSE = True
# =================================================

def find_backup_files(directory, keywords, recursive=True):
    """
    查找所有包含备份标识的文件
    """
    directory = Path(directory)
    backup_files = []
    
    if recursive:
        # 递归查找所有文件
        all_files = list(directory.rglob("*"))
    else:
        # 仅当前目录
        all_files = list(directory.glob("*"))
    
    for file_path in all_files:
        if file_path.is_file():
            # 检查文件名是否包含任何备份标识
            file_name_lower = file_path.name.lower()
            for keyword in keywords:
                if keyword.lower() in file_name_lower:
                    backup_files.append(file_path)
                    break
    
    return backup_files

def get_file_size_safe(file_path):
    """
    安全获取文件大小，如果文件不存在则返回0
    """
    try:
        if file_path.exists():
            return file_path.stat().st_size
        else:
            return 0
    except:
        return 0

def delete_backup_files(backup_files, confirm=True, verbose=True):
    """
    删除备份文件
    """
    deleted_files = []
    failed_files = []
    skipped_files = []
    
    if not backup_files:
        print("未找到任何备份文件")
        return deleted_files, failed_files, skipped_files
    
    # 先过滤掉已经不存在的文件
    existing_files = []
    for file_path in backup_files:
        if file_path.exists():
            existing_files.append(file_path)
        else:
            skipped_files.append(file_path)
            if verbose:
                print(f"  ⚠ 文件已不存在: {file_path.name}")
    
    if skipped_files:
        print(f"\n跳过 {len(skipped_files)} 个已不存在的文件")
    
    if not existing_files:
        print("没有找到需要删除的备份文件（所有备份文件可能已被删除）")
        return deleted_files, failed_files, skipped_files
    
    print(f"\n找到 {len(existing_files)} 个备份文件:")
    for i, file_path in enumerate(existing_files, 1):
        file_size = round(get_file_size_safe(file_path) / 1024, 2)  # KB
        print(f"  {i}. {file_path.name} ({file_size} KB)")
        if verbose and file_path.parent != Path(DATA_DIR):
            print(f"     路径: {file_path.parent}")
    
    if not confirm:
        print("\n[预览模式] 未执行删除操作")
        print("如需删除，请设置 CONFIRM_DELETE = True")
        return deleted_files, failed_files, skipped_files
    
    print("\n开始删除备份文件...")
    
    for file_path in existing_files:
        try:
            file_size = round(get_file_size_safe(file_path) / 1024, 2)
            file_path.unlink()  # 删除文件
            deleted_files.append(file_path)
            if verbose:
                print(f"  ✓ 已删除: {file_path.name} ({file_size} KB)")
        except Exception as e:
            failed_files.append((file_path, str(e)))
            if verbose:
                print(f"  ✗ 删除失败: {file_path.name} - {e}")
    
    return deleted_files, failed_files, skipped_files

# 开始执行
print("=" * 100)
print("删除备份文件")
print(f"执行时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 100)

# 检查目录
target_dir = Path(DATA_DIR)
print(f"\n目标目录: {target_dir}")
print(f"目录是否存在: {target_dir.exists()}")

if not target_dir.exists():
    raise FileNotFoundError(f"目录不存在: {target_dir}")

print(f"\n搜索配置:")
print(f"  - 备份标识: {BACKUP_KEYWORDS}")
print(f"  - 递归搜索: {RECURSIVE}")
print(f"  - 确认删除: {CONFIRM_DELETE}")

# 查找备份文件
backup_files = find_backup_files(target_dir, BACKUP_KEYWORDS, RECURSIVE)

# 显示统计信息
print(f"\n查找结果:")
print(f"  - 找到备份文件: {len(backup_files)} 个")

if backup_files:
    # 计算总大小（仅计算存在的文件）
    total_size = sum(get_file_size_safe(f) for f in backup_files)
    total_size_mb = round(total_size / 1024 / 1024, 2)
    print(f"  - 总文件大小: {total_size_mb} MB")
    
    # 删除备份文件
    deleted_files, failed_files, skipped_files = delete_backup_files(backup_files, CONFIRM_DELETE, VERBOSE)
    
    # 显示删除结果
    print("\n" + "=" * 100)
    print("删除结果汇总")
    print("=" * 100)
    print(f"成功删除: {len(deleted_files)} 个文件")
    print(f"删除失败: {len(failed_files)} 个文件")
    print(f"跳过（文件已不存在）: {len(skipped_files)} 个文件")
    
    if deleted_files:
        # 计算释放的空间（仅计算成功删除的文件）
        freed_size = sum(get_file_size_safe(f) for f in deleted_files)
        freed_size_mb = round(freed_size / 1024 / 1024, 2)
        if CONFIRM_DELETE:
            print(f"释放空间: {freed_size_mb} MB")
        else:
            print(f"预计释放空间: {freed_size_mb} MB")
    
    if failed_files:
        print("\n删除失败的文件:")
        for file_path, error in failed_files:
            print(f"  - {file_path.name}: {error}")
    
    if skipped_files and VERBOSE:
        print("\n跳过的文件（已不存在）:")
        for file_path in skipped_files[:10]:  # 只显示前10个
            print(f"  - {file_path.name}")
        if len(skipped_files) > 10:
            print(f"  ... 还有 {len(skipped_files) - 10} 个文件")
else:
    print("\n未找到任何备份文件")

# 可选：保存操作日志
if CONFIRM_DELETE and (deleted_files or failed_files):
    log_lines = []
    log_lines.append("=" * 100)
    log_lines.append(f"删除备份文件日志")
    log_lines.append(f"执行时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    log_lines.append("=" * 100)
    log_lines.append(f"\n目标目录: {DATA_DIR}")
    log_lines.append(f"备份标识: {BACKUP_KEYWORDS}")
    log_lines.append(f"\n删除结果:")
    log_lines.append(f"  - 成功删除: {len(deleted_files)} 个")
    log_lines.append(f"  - 删除失败: {len(failed_files)} 个")
    log_lines.append(f"  - 跳过（已不存在）: {len(skipped_files)} 个")
    
    if deleted_files:
        freed_size = sum(get_file_size_safe(f) for f in deleted_files)
        freed_size_mb = round(freed_size / 1024 / 1024, 2)
        log_lines.append(f"  - 释放空间: {freed_size_mb} MB")
        log_lines.append(f"\n已删除文件列表:")
        for file_path in deleted_files:
            file_size = round(get_file_size_safe(file_path) / 1024, 2)
            log_lines.append(f"  - {file_path.name} ({file_size} KB)")
    
    if failed_files:
        log_lines.append(f"\n删除失败文件列表:")
        for file_path, error in failed_files:
            log_lines.append(f"  - {file_path.name}: {error}")
    
    # 保存日志
    log_dir = Path(os.getcwd())
    log_filename = f"backup_deletion_log_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"
    log_filepath = log_dir / log_filename
    
    with open(log_filepath, 'w', encoding='utf-8') as f:
        f.write('\n'.join(log_lines))
    
    print(f"\n操作日志已保存至: {log_filepath}")

print("\n" + "=" * 100)
print("操作完成")
print("=" * 100)

删除备份文件
执行时间: 2026-05-11 17:37:05

目标目录: /data/SJJ/UBT/lerobot_0.5.1/datasets/Part_Sorting/test/data/chunk-000
目录是否存在: True

搜索配置:
  - 备份标识: ['_backup', '.backup', '_bak', '.bak', 'backup_', 'back_up', 'parquet_backup']
  - 递归搜索: True
  - 确认删除: True

查找结果:
  - 找到备份文件: 1 个
  - 总文件大小: 0.2 MB

找到 1 个备份文件:
  1. episode_000000.parquet_backup (207.67 KB)

开始删除备份文件...
  ✓ 已删除: episode_000000.parquet_backup (207.67 KB)

删除结果汇总
成功删除: 1 个文件
删除失败: 0 个文件
跳过（文件已不存在）: 0 个文件
释放空间: 0.0 MB

操作日志已保存至: /data/SJJ/UBT/lerobot_0.5.1/datasets/backup_deletion_log_20260511_173705.txt

操作完成


## 查看夹爪数据

In [8]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
from datetime import datetime

PROJECT_ROOT = Path.cwd().parent.resolve()

# ========== 全局变量定义（已修正）==========
# 你指定的 parquet 文件
DATA_FILE = str(PROJECT_ROOT / "datasets/Part_Sorting/26_5_11_episode_1000_obj_1_v3/data/chunk-000/file-000.parquet")

# 【重要】提取数组内部的第 18~20 个元素（索引）
ARRAY_SLICE = slice(18, 20)

# 要查看的列
TARGET_COLUMNS = ["observation.state", "action"]

# 是否保存为txt文件
SAVE_TO_TXT = True

# 输出目录
OUTPUT_DIR = os.getcwd()
TXT_PREFIX = "array_slice_18-20"
# =================================================

output_lines = []

def print_and_log(text):
    print(text)
    if SAVE_TO_TXT:
        output_lines.append(text)

# ================== 主程序 ==================
print_and_log("=" * 80)
print_and_log(f"LeRobot 数组切片提取")
print_and_log(f"文件: {DATA_FILE}")
print_and_log(f"目标列: {TARGET_COLUMNS}")
print_and_log(f"提取数组内部索引: [18:20]")
print_and_log(f"生成时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print_and_log("=" * 80)

# 读取文件
file_path = Path(DATA_FILE)
if not file_path.exists():
    raise FileNotFoundError(f"文件不存在: {file_path}")

df = pd.read_parquet(file_path)
print_and_log(f"\n数据读取完成，共 {len(df)} 行")

# 检查列
for col in TARGET_COLUMNS:
    if col not in df.columns:
        raise KeyError(f"缺少列: {col}，可用列: {list(df.columns)}")

# 提取前5行（你要的前5行数据）
print_and_log("\n📌 提取 前5行 数据中 observation.state 和 action 的数组 [18:20] 元素")
print_and_log("-" * 80)

df_head5 = df.head(5)

for idx in df_head5.index:
    print_and_log(f"\n=== 第 {idx} 行 ===")
    
    # observation.state
    state_arr = df.loc[idx, "observation.state"]
    state_slice = state_arr[ARRAY_SLICE]
    print_and_log(f"observation.state [18:20] = {state_slice}")
    
    # action
    action_arr = df.loc[idx, "action"]
    action_slice = action_arr[ARRAY_SLICE]
    print_and_log(f"action           [18:20] = {action_slice}")

# ================== 【新增功能】提取所有唯一值 ==================
print_and_log("\n\n" + "="*80)
print_and_log("📊 全量数据筛选：observation.state & action 数组 [18:20] 唯一值（unique）")
print_and_log("="*80)

# 存储所有切片结果
all_state_slices = []
all_action_slices = []

for idx in df.index:
    # state
    state_arr = df.loc[idx, "observation.state"]
    state_slice = state_arr[ARRAY_SLICE]
    all_state_slices.append(tuple(state_slice))  # tuple 才能去重
    
    # action
    action_arr = df.loc[idx, "action"]
    action_slice = action_arr[ARRAY_SLICE]
    all_action_slices.append(tuple(action_slice))

# 去重
unique_state = sorted(list(set(all_state_slices)))
unique_action = sorted(list(set(all_action_slices)))

# 输出
print_and_log(f"\n✅ observation.state [18:20] 唯一值（共 {len(unique_state)} 种）：")
for v in unique_state:
    print_and_log(f"  {v}")

print_and_log(f"\n✅ action           [18:20] 唯一值（共 {len(unique_action)} 种）：")
for v in unique_action:
    print_and_log(f"  {v}")

# =================================================================

# 保存 TXT
if SAVE_TO_TXT:
    output_dir = Path(OUTPUT_DIR)
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    txt_path = output_dir / f"{TXT_PREFIX}_{timestamp}.txt"
    
    with open(txt_path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(output_lines))
    
    print_and_log("\n" + "="*80)
    print_and_log(f"✅ 已保存到: {txt_path}")
    print_and_log("="*80)

print_and_log("\n🎉 提取完成！")


LeRobot 数组切片提取
文件: /data/SJJ/UBT/lerobot_0.5.1/datasets/Part_Sorting/26_5_11_episode_1000_obj_1_v3/data/chunk-000/file-000.parquet
目标列: ['observation.state', 'action']
提取数组内部索引: [18:20]
生成时间: 2026-05-13 16:56:51

数据读取完成，共 192000 行

📌 提取 前5行 数据中 observation.state 和 action 的数组 [18:20] 元素
--------------------------------------------------------------------------------

=== 第 0 行 ===
observation.state [18:20] = [-1. -1.]
action           [18:20] = [-1. -1.]

=== 第 1 行 ===
observation.state [18:20] = [-1. -1.]
action           [18:20] = [-1. -1.]

=== 第 2 行 ===
observation.state [18:20] = [-1. -1.]
action           [18:20] = [-1. -1.]

=== 第 3 行 ===
observation.state [18:20] = [-1. -1.]
action           [18:20] = [-1. -1.]

=== 第 4 行 ===
observation.state [18:20] = [-1. -1.]
action           [18:20] = [-1. -1.]


📊 全量数据筛选：observation.state & action 数组 [18:20] 唯一值（unique）

✅ observation.state [18:20] 唯一值（共 4 种）：
  (-1.0, -1.0)
  (-1.0, 1.0)
  (1.0, -1.0)
  (1.0, 1.0)

✅ action           [18:

## 统计 action 数组第18、19号元素的值分布

遍历数据集中所有行，统计 `action` 列数组在索引 18 和 19 处的值种类及出现次数。用于分析夹爪动作的取值模式。

**使用示例：**
- 修改 `DATA_FILE` 为目标 parquet 文件路径
- 修改 `POSITIONS` 为需要分析的数组索引列表
- 运行后查看各位置的值分布

In [6]:
# Jupyter Notebook 单元格代码
import pandas as pd
import numpy as np
from collections import Counter
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.resolve()

# ========== 配置（只看 18:19）==========
DATA_FILE = str(PROJECT_ROOT / "datasets/Part_Sorting/single/data/chunk-000/file-000.parquet")
# 只查看数组索引 18 和 19
POSITIONS = [18, 19]
# 只查看 action 列
TARGET_COL = "action"
# ======================================

output_lines = []
def print_and_log(text):
    print(text)
    output_lines.append(text)

# 读取数据
print_and_log("="*60)
print_and_log("🔍 分析 action 数组 第18、19号元素 的值种类")
print_and_log("="*60)

df = pd.read_parquet(DATA_FILE)
print_and_log(f"数据集总行数：{len(df)}")

# 存储结果
results = {18: [], 19: []}

# 遍历所有数据
for idx in range(len(df)):
    arr = df[TARGET_COL].iloc[idx]
    val18 = arr[18]
    val19 = arr[19]
    results[18].append(val18)
    results[19].append(val19)

# 统计输出
print_and_log("\n" + "="*50)
print_and_log("📊 统计结果（action 列）")
print_and_log("="*50)

for pos in POSITIONS:
    values = results[pos]
    counter = Counter(values)
    
    print_and_log(f"\n✅ 第 {pos} 号元素：")
    print_and_log(f"   出现的值：{list(counter.keys())}")
    for val, cnt in counter.items():
        print_and_log(f"   → {val}  共 {cnt} 次")

# 保存到 txt
txt_path = Path.cwd() / "action_18-19_value_types.txt"
with open(txt_path, 'w', encoding='utf-8') as f:
    f.write('\n'.join(output_lines))

print_and_log(f"\n✅ 结果已保存到：{txt_path}")
print_and_log("\n🎉 完成！")


🔍 分析 action 数组 第18、19号元素 的值种类
数据集总行数：79344

📊 统计结果（action 列）

✅ 第 18 号元素：
   出现的值：[-1.0]
   → -1.0  共 79344 次

✅ 第 19 号元素：
   出现的值：[-1.0, 1.0]
   → -1.0  共 45144 次
   → 1.0  共 34200 次

✅ 结果已保存到：/data/SJJ/UBT/lerobot_0.5.1/datasets/action_18-19_value_types.txt

🎉 完成！


## 查看meta目录下所有.jsonl文件

In [3]:
# Jupyter Notebook 单元格代码
import os
from pathlib import Path
import json
from datetime import datetime
from collections import defaultdict
import numpy as np

PROJECT_ROOT = Path.cwd().parent.resolve()

# ========== 全局变量定义 ==========
# 数据目录路径
DATA_DIR = str(PROJECT_ROOT / "datasets/Part_Sorting/test/meta")

# 是否保存为txt文件（True: 保存, False: 不保存）
SAVE_TO_TXT = True

# txt文件输出目录（None表示保存到当前工作目录）
OUTPUT_DIR = None

# txt文件名前缀
TXT_PREFIX = "jsonl_schema_analysis"

# 是否递归搜索子目录
RECURSIVE = True

# 是否显示每个文件的详细信息
SHOW_PER_FILE_DETAILS = True

# 是否显示嵌套路径（如 data.info.id）
SHOW_NESTED_PATHS = True

# 分隔符用于嵌套路径
NESTED_SEPARATOR = '.'

# 采样行数用于推断类型和维度（-1表示所有行，否则采样指定行数）
SAMPLE_LINES = 100
# =================================================

def get_value_type_and_shape(value):
    """
    获取值的类型和维度信息
    返回: (type_name, shape_info)
    """
    if isinstance(value, dict):
        return "dict", f"keys: {list(value.keys())[:5]}..."
    elif isinstance(value, list):
        shape = f"len={len(value)}"
        if len(value) > 0:
            # 检查第一个元素的类型
            first_type = type(value[0]).__name__
            if isinstance(value[0], (list, dict)):
                # 对于嵌套结构，进一步分析
                if isinstance(value[0], list):
                    shape += f", elements: list of {first_type}"
                elif isinstance(value[0], dict):
                    shape += f", elements: dict with {len(value[0].keys())} keys"
            else:
                shape += f", elements: {first_type}"
                
            # 对于数值数组，显示范围
            if len(value) > 0 and all(isinstance(x, (int, float, np.number)) for x in value[:10]):
                try:
                    min_val = min(value[:10])
                    max_val = max(value[:10])
                    shape += f", range: [{min_val:.4f}, {max_val:.4f}]"
                except:
                    pass
        return "list", shape
    elif isinstance(value, (int, np.integer)):
        return "int", str(value)[:20]
    elif isinstance(value, (float, np.floating)):
        return "float", f"{value:.6f}" if isinstance(value, float) else str(value)[:20]
    elif isinstance(value, bool):
        return "bool", str(value)
    elif isinstance(value, str):
        return "str", f"len={len(value)}" if len(value) > 50 else value[:50]
    elif value is None:
        return "None", "null"
    else:
        return type(value).__name__, str(value)[:50]

def extract_keys_with_schema(obj, parent_key='', separator='.'):
    """
    递归提取JSON对象中的所有键，并记录每个键的数据类型和维度信息
    返回字典: {key_path: {'type': type_name, 'shape': shape_info, 'example': example_value}}
    """
    schema_info = {}
    
    if isinstance(obj, dict):
        for key, value in obj.items():
            # 构建完整的键路径
            current_key = f"{parent_key}{separator}{key}" if parent_key else key
            
            # 获取值的信息
            type_name, shape_info = get_value_type_and_shape(value)
            schema_info[current_key] = {
                'type': type_name,
                'shape': shape_info,
                'example': value if type_name in ['dict', 'list'] else None
            }
            
            # 递归提取嵌套键
            if isinstance(value, (dict, list)):
                nested_schema = extract_keys_with_schema(value, current_key, separator)
                schema_info.update(nested_schema)
    
    elif isinstance(obj, list):
        # 对于列表，检查第一个元素的结构（假设列表内结构一致）
        if obj and len(obj) > 0:
            first_item = obj[0]
            if isinstance(first_item, (dict, list)):
                nested_schema = extract_keys_with_schema(first_item, parent_key, separator)
                schema_info.update(nested_schema)
    
    return schema_info

def merge_schema_info(schema_dict, new_schema):
    """
    合并多个schema信息，保留类型信息和示例
    """
    for key, value in new_schema.items():
        if key not in schema_dict:
            schema_dict[key] = value
        else:
            # 如果类型不同，标记为混合类型
            if schema_dict[key]['type'] != value['type']:
                schema_dict[key]['type'] = f"mixed({schema_dict[key]['type']}/{value['type']})"

def analyze_jsonl_file(file_path, sample_lines=SAMPLE_LINES):
    """
    分析单个jsonl文件，提取所有键及其类型和维度信息
    """
    file_info = {
        'path': file_path,
        'name': file_path.name,
        'size_mb': round(file_path.stat().st_size / 1024 / 1024, 2),
        'line_count': 0,
        'schema': {},
        'error': None
    }
    
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            lines = f.readlines()
        
        file_info['line_count'] = len(lines)
        
        # 确定采样行数
        total_lines = len(lines)
        if sample_lines == -1 or sample_lines > total_lines:
            lines_to_analyze = lines
        else:
            lines_to_analyze = lines[:sample_lines]
        
        # 解析每一行，提取schema信息
        merged_schema = {}
        
        for line_num, line in enumerate(lines_to_analyze, 1):
            if line.strip():
                try:
                    data = json.loads(line)
                    schema = extract_keys_with_schema(data, separator=NESTED_SEPARATOR)
                    merge_schema_info(merged_schema, schema)
                    
                except json.JSONDecodeError as e:
                    if file_info['error'] is None:
                        file_info['error'] = f"第{line_num}行JSON解析错误: {e}"
                    continue
        
        file_info['schema'] = merged_schema
        
    except Exception as e:
        file_info['error'] = str(e)
    
    return file_info

def format_file_info(info, show_details=True):
    """
    格式化文件信息为文本
    """
    lines = []
    lines.append("=" * 100)
    lines.append(f"文件: {info['name']}")
    lines.append(f"路径: {info['path']}")
    lines.append(f"大小: {info['size_mb']} MB")
    lines.append(f"行数: {info['line_count']:,}")
    
    if info['error']:
        lines.append(f"错误: {info['error']}")
        lines.append("=" * 100)
        lines.append("")
        return lines
    
    lines.append(f"唯一键总数: {len(info['schema'])}")
    
    if show_details and info['schema']:
        lines.append(f"\n键的详细信息（类型和维度）:")
        lines.append(f"{'序号':<4} {'键路径':<50} {'类型':<12} {'维度/示例'}")
        lines.append("-" * 100)
        
        for i, (key, value) in enumerate(sorted(info['schema'].items()), 1):
            type_name = value['type']
            shape_info = value['shape']
            # 截断过长的键路径
            display_key = key if len(key) <= 45 else key[:42] + "..."
            lines.append(f"{i:<4} {display_key:<50} {type_name:<12} {shape_info}")
    
    lines.append("=" * 100)
    lines.append("")
    return lines

# 开始分析
print("=" * 100)
print("JSONL文件结构分析报告（键、类型、维度）")
print(f"生成时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 100)

# 检查目录
target_dir = Path(DATA_DIR)
print(f"\n目标目录: {target_dir}")
print(f"目录是否存在: {target_dir.exists()}")

if not target_dir.exists():
    raise FileNotFoundError(f"目录不存在: {target_dir}")

# 查找所有.jsonl文件
if RECURSIVE:
    jsonl_files = sorted(target_dir.rglob("*.jsonl"))
else:
    jsonl_files = sorted(target_dir.glob("*.jsonl"))

print(f"\n搜索配置:")
print(f"  - 递归搜索: {RECURSIVE}")
print(f"  - 采样行数: {SAMPLE_LINES if SAMPLE_LINES > 0 else '所有行'}")
print(f"  - 显示嵌套路径: {SHOW_NESTED_PATHS}")
print(f"  - 嵌套分隔符: '{NESTED_SEPARATOR}'")

print(f"\n找到 {len(jsonl_files)} 个 .jsonl 文件")

if len(jsonl_files) == 0:
    print(f"目录中没有找到任何 .jsonl 文件")
else:
    # 存储所有输出内容
    output_lines = []
    
    # 存储所有文件信息
    all_files_info = []
    
    # 全局schema（合并所有文件）
    global_schema = {}
    
    print("\n开始分析JSONL文件...")
    
    for i, file_path in enumerate(jsonl_files, 1):
        print(f"\n[{i}/{len(jsonl_files)}] 正在分析: {file_path.name}")
        
        # 分析文件
        file_info = analyze_jsonl_file(file_path)
        all_files_info.append(file_info)
        
        # 更新全局schema
        merge_schema_info(global_schema, file_info['schema'])
        
        # 格式化输出
        if SHOW_PER_FILE_DETAILS:
            formatted_info = format_file_info(file_info, SHOW_PER_FILE_DETAILS)
            for line in formatted_info:
                print(line)
                if SAVE_TO_TXT:
                    output_lines.append(line)
        else:
            # 只显示基本信息
            print(f"  大小: {file_info['size_mb']} MB, 行数: {file_info['line_count']:,}, 键数: {len(file_info['schema'])}")
            if SAVE_TO_TXT:
                output_lines.append(f"文件: {file_info['name']}")
                output_lines.append(f"  大小: {file_info['size_mb']} MB, 行数: {file_info['line_count']:,}, 键数: {len(file_info['schema'])}")
    
    # 显示全局汇总信息
    print("\n" + "=" * 100)
    print("全局汇总信息")
    print("=" * 100)
    
    total_files = len(all_files_info)
    total_lines = sum(info['line_count'] for info in all_files_info if not info['error'])
    total_size_mb = sum(info['size_mb'] for info in all_files_info)
    error_files = [info for info in all_files_info if info['error']]
    
    print(f"总文件数: {total_files}")
    print(f"成功读取: {total_files - len(error_files)}")
    print(f"读取失败: {len(error_files)}")
    print(f"总行数: {total_lines:,}")
    print(f"总大小: {round(total_size_mb, 2)} MB")
    print(f"全局唯一键总数: {len(global_schema)}")
    
    # 显示全局schema
    if global_schema:
        print(f"\n所有JSONL文件中出现的键及其类型/维度（共{len(global_schema)}个）:")
        print(f"{'序号':<4} {'键路径':<50} {'类型':<12} {'维度/示例'}")
        print("-" * 100)
        
        # 按根键分组显示
        root_keys = {}
        for key, value in global_schema.items():
            root = key.split(NESTED_SEPARATOR)[0] if NESTED_SEPARATOR in key else key
            if root not in root_keys:
                root_keys[root] = []
            root_keys[root].append((key, value))
        
        print(f"\n按根键分组显示:")
        for root in sorted(root_keys.keys()):
            print(f"\n  [{root}] ({len(root_keys[root])} 个子键):")
            for key, value in sorted(root_keys[root]):
                indent = "    " + "  " * (key.count(NESTED_SEPARATOR))
                print(f"{indent}- {key}")
                print(f"{indent}  类型: {value['type']}, {value['shape']}")
    
    # 显示每个文件的统计
    print(f"\n各文件统计:")
    for info in all_files_info:
        if not info['error']:
            print(f"  - {info['name']}: {len(info['schema'])} 个唯一键")
    
    # 显示有错误的文件
    if error_files:
        print(f"\n错误文件列表:")
        for info in error_files:
            print(f"  - {info['name']}: {info['error']}")
    
    # 汇总信息添加到日志
    if SAVE_TO_TXT:
        summary_lines = [
            "\n" + "=" * 100,
            "全局汇总信息",
            "=" * 100,
            f"总文件数: {total_files}",
            f"成功读取: {total_files - len(error_files)}",
            f"读取失败: {len(error_files)}",
            f"总行数: {total_lines:,}",
            f"总大小: {round(total_size_mb, 2)} MB",
            f"全局唯一键总数: {len(global_schema)}",
        ]
        
        if global_schema:
            summary_lines.append(f"\n所有JSONL文件中出现的键及其类型/维度（共{len(global_schema)}个）:")
            summary_lines.append(f"{'序号':<4} {'键路径':<50} {'类型':<12} {'维度/示例'}")
            summary_lines.append("-" * 100)
            
            for i, (key, value) in enumerate(sorted(global_schema.items()), 1):
                display_key = key if len(key) <= 45 else key[:42] + "..."
                summary_lines.append(f"{i:<4} {display_key:<50} {value['type']:<12} {value['shape']}")
            
            # 添加分组显示
            summary_lines.append(f"\n按根键分组显示:")
            root_keys = {}
            for key, value in global_schema.items():
                root = key.split(NESTED_SEPARATOR)[0] if NESTED_SEPARATOR in key else key
                if root not in root_keys:
                    root_keys[root] = []
                root_keys[root].append((key, value))
            
            for root in sorted(root_keys.keys()):
                summary_lines.append(f"\n  [{root}] ({len(root_keys[root])} 个子键):")
                for key, value in sorted(root_keys[root]):
                    indent = "    " + "  " * (key.count(NESTED_SEPARATOR))
                    summary_lines.append(f"{indent}- {key}")
                    summary_lines.append(f"{indent}  类型: {value['type']}, {value['shape']}")
        
        if error_files:
            summary_lines.append(f"\n错误文件列表:")
            for info in error_files:
                summary_lines.append(f"  - {info['name']}: {info['error']}")
        
        output_lines.extend(summary_lines)
        
        # 保存为txt文件
        if OUTPUT_DIR is None:
            output_dir = Path(os.getcwd())
        else:
            output_dir = Path(OUTPUT_DIR)
            output_dir.mkdir(parents=True, exist_ok=True)
        
        # 生成文件名
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        dir_name = target_dir.name
        txt_filename = f"{TXT_PREFIX}_{dir_name}_{timestamp}.txt"
        txt_filepath = output_dir / txt_filename
        
        # 写入文件
        with open(txt_filepath, 'w', encoding='utf-8') as f:
            f.write('\n'.join(output_lines))
        
        print(f"\n分析报告已保存至: {txt_filepath}")
        print(f"保存目录: {output_dir.absolute()}")
    else:
        print("\n未保存txt文件（SAVE_TO_TXT = False）")

print("\n" + "=" * 100)
print("分析完成")
print("=" * 100)

JSONL文件结构分析报告（键、类型、维度）
生成时间: 2026-05-11 17:37:42

目标目录: /data/SJJ/UBT/lerobot_0.5.1/datasets/Part_Sorting/test/meta
目录是否存在: True

搜索配置:
  - 递归搜索: True
  - 采样行数: 100
  - 显示嵌套路径: True
  - 嵌套分隔符: '.'

找到 3 个 .jsonl 文件

开始分析JSONL文件...

[1/3] 正在分析: episodes.jsonl
文件: episodes.jsonl
路径: /data/SJJ/UBT/lerobot_0.5.1/datasets/Part_Sorting/test/meta/episodes.jsonl
大小: 0.0 MB
行数: 1
唯一键总数: 3

键的详细信息（类型和维度）:
序号   键路径                                                类型           维度/示例
----------------------------------------------------------------------------------------------------
1    episode_index                                      int          0
2    length                                             int          768
3    tasks                                              list         len=1, elements: str


[2/3] 正在分析: episodes_stats.jsonl
文件: episodes_stats.jsonl
路径: /data/SJJ/UBT/lerobot_0.5.1/datasets/Part_Sorting/test/meta/episodes_stats.jsonl
大小: 0.01 MB
行数: 1
唯一键总数: 74

键的详细信息（类型和维度）:
序号

## 对meta目录下的jsonl文件进行分别处理

In [5]:
# Jupyter Notebook 单元格代码
import os
from pathlib import Path
import json
from datetime import datetime
import numpy as np

PROJECT_ROOT = Path.cwd().parent.resolve()

# ========== 全局变量定义 ==========
# 数据目录路径
DATA_DIR = str(PROJECT_ROOT / "datasets/Part_Sorting/26_5_11_episode_1000_obj_4/meta")

# ========== 处理选项 ==========
# 是否处理 episodes.jsonl 文件
PROCESS_EPISODES = True

# 是否处理 episodes_stats.jsonl 文件
PROCESS_EPISODES_STATS = True

# 是否处理 tasks.jsonl 文件
PROCESS_TASKS = True

# ========== 其他配置 ==========
# 是否创建备份（True: 备份原始文件, False: 不备份）
CREATE_BACKUP = False

# 备份文件后缀
BACKUP_SUFFIX = "_backup"

# 是否覆盖原文件（False: 保存到新文件）
OVERWRITE_ORIGINAL = True

# 输出目录（当OVERWRITE_ORIGINAL=False时使用）
OUTPUT_DIR = None

# 是否保存处理日志
SAVE_LOG = False
# =================================================

def process_episodes_file(data, file_path):
    """
    处理episodes.jsonl文件
    将tasks键的每个值替换为"Part Sorting"
    """
    modified = False
    for item in data:
        if 'tasks' in item:
            original_tasks = item['tasks']
            # 将所有task替换为"Part Sorting"
            new_tasks = ["Part Sorting"] * len(original_tasks)
            item['tasks'] = new_tasks
            modified = True
            print(f"      修改tasks: {len(original_tasks)}个任务 -> Part Sorting")
    return modified

def process_episodes_stats_file(data, file_path):
    """
    处理episodes_stats.jsonl文件
    将observation.environment_state的统计量数组拼接到observation.state对应的统计量数组后面
    注意：count不处理，直接删除observation.environment_state
    """
    modified = False
    
    for idx, item in enumerate(data):
        # 检查是否有observation.environment_state和observation.state键
        if 'observation.environment_state' not in item['stats'] or 'observation.state' not in item['stats']:
            continue
            
        env_stats = item['stats']['observation.environment_state']
        state_stats = item['stats']['observation.state']
        
        print(f"      处理第{idx+1}条数据...")
        print(f"        observation.state统计量键: {list(state_stats.keys())}")
        print(f"        observation.environment_state统计量键: {list(env_stats.keys())}")
        
        # 需要拼接的统计字段（不包括count）
        merge_fields = ['min', 'max', 'mean', 'std']
        
        # 拼接统计量数组
        for field in merge_fields:
            if field in env_stats and field in state_stats:
                state_val = state_stats[field]
                env_val = env_stats[field]
                
                # 检查是否都是列表类型
                if isinstance(state_val, list) and isinstance(env_val, list):
                    original_len = len(state_val)
                    # 拼接数组
                    state_stats[field] = state_val + env_val
                    print(f"        - {field}: {original_len}维 + {len(env_val)}维 -> {len(state_stats[field])}维")
                    modified = True
                else:
                    print(f"        - {field}: 警告 - 不是列表类型 (state类型={type(state_val)}, env类型={type(env_val)})")
        
        # 删除observation.environment_state
        if 'observation.environment_state' in item['stats']:
            del item['stats']['observation.environment_state']
            print(f"        - 已删除 observation.environment_state")
            modified = True
    
    return modified

def process_tasks_file(data, file_path):
    """
    处理tasks.jsonl文件
    将task键的值替换为"Part Sorting"
    """
    modified = False
    for item in data:
        if 'task' in item:
            original_task = item['task']
            item['task'] = "Part Sorting"
            modified = True
            print(f"      修改task: {original_task} -> Part Sorting")
    return modified

def process_jsonl_file(file_path, process_type, create_backup=True, overwrite=True, output_dir=None):
    """
    处理单个jsonl文件
    process_type: 'episodes', 'episodes_stats', 'tasks'
    """
    file_info = {
        'path': file_path,
        'name': file_path.name,
        'size_mb': round(file_path.stat().st_size / 1024 / 1024, 2),
        'line_count': 0,
        'success': False,
        'modified': False,
        'error': None
    }
    
    try:
        print(f"\n处理文件: {file_path.name}")
        
        # 读取所有行
        with open(file_path, 'r', encoding='utf-8') as f:
            lines = f.readlines()
        
        file_info['line_count'] = len(lines)
        print(f"  原始行数: {file_info['line_count']}")
        
        # 解析JSON数据
        data = []
        for line_num, line in enumerate(lines, 1):
            if line.strip():
                try:
                    data.append(json.loads(line))
                except json.JSONDecodeError as e:
                    print(f"  警告: 第{line_num}行JSON解析错误: {e}")
                    file_info['error'] = f"第{line_num}行解析错误"
                    return file_info
        
        # 根据类型选择处理函数
        if process_type == 'episodes':
            print(f"  处理类型: episodes文件")
            modified = process_episodes_file(data, file_path)
        elif process_type == 'episodes_stats':
            print(f"  处理类型: episodes_stats文件")
            modified = process_episodes_stats_file(data, file_path)
        elif process_type == 'tasks':
            print(f"  处理类型: tasks文件")
            modified = process_tasks_file(data, file_path)
        else:
            print(f"  跳过: 未知的处理类型")
            file_info['success'] = True
            return file_info
        
        file_info['modified'] = modified
        
        if not modified:
            print(f"  无需修改")
            file_info['success'] = True
            return file_info
        
        # 保存修改后的数据
        if overwrite:
            # 如果需要备份，先创建备份
            if create_backup:
                backup_path = file_path.with_suffix(f'.jsonl{BACKUP_SUFFIX}')
                if not backup_path.exists():
                    # 备份原始内容
                    with open(backup_path, 'w', encoding='utf-8') as f:
                        f.writelines(lines)
                    print(f"  已创建备份: {backup_path.name}")
                else:
                    print(f"  备份文件已存在: {backup_path.name}")
            
            # 覆盖原文件
            with open(file_path, 'w', encoding='utf-8') as f:
                for item in data:
                    f.write(json.dumps(item, ensure_ascii=False) + '\n')
            print(f"  已覆盖原文件: {file_path.name}")
        else:
            # 保存到新文件
            if output_dir is None:
                output_path = file_path.parent / f"{file_path.stem}_processed.jsonl"
            else:
                output_path = Path(output_dir) / f"{file_path.stem}_processed.jsonl"
                output_path.parent.mkdir(parents=True, exist_ok=True)
            
            with open(output_path, 'w', encoding='utf-8') as f:
                for item in data:
                    f.write(json.dumps(item, ensure_ascii=False) + '\n')
            print(f"  已保存到新文件: {output_path}")
        
        file_info['success'] = True
        
    except Exception as e:
        file_info['error'] = str(e)
        print(f"  错误: {e}")
    
    return file_info

# 开始处理
print("=" * 100)
print("JSONL文件处理 - episodes_stats统计量数组拼接")
print(f"开始时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 100)

# 检查目录
target_dir = Path(DATA_DIR)
print(f"\n目标目录: {target_dir}")
print(f"目录是否存在: {target_dir.exists()}")

if not target_dir.exists():
    raise FileNotFoundError(f"目录不存在: {target_dir}")

# 显示处理配置
print(f"\n处理配置:")
print(f"  - 处理 episodes.jsonl: {PROCESS_EPISODES}")
print(f"  - 处理 episodes_stats.jsonl: {PROCESS_EPISODES_STATS}")
print(f"  - 处理 tasks.jsonl: {PROCESS_TASKS}")
print(f"  - 创建备份: {CREATE_BACKUP}")
print(f"  - 覆盖原文件: {OVERWRITE_ORIGINAL}")
print(f"  - 备份后缀: {BACKUP_SUFFIX}")
if not OVERWRITE_ORIGINAL:
    print(f"  - 输出目录: {OUTPUT_DIR if OUTPUT_DIR else '原文件目录'}")

# 查找需要处理的文件
files_to_process = []

if PROCESS_EPISODES:
    episodes_files = list(target_dir.rglob("episodes.jsonl"))
    files_to_process.extend([(f, 'episodes') for f in episodes_files if BACKUP_SUFFIX not in f.suffixes])
    print(f"\n找到 {len(episodes_files)} 个 episodes.jsonl 文件")

if PROCESS_EPISODES_STATS:
    stats_files = list(target_dir.rglob("episodes_stats.jsonl"))
    files_to_process.extend([(f, 'episodes_stats') for f in stats_files if BACKUP_SUFFIX not in f.suffixes])
    print(f"找到 {len(stats_files)} 个 episodes_stats.jsonl 文件")

if PROCESS_TASKS:
    tasks_files = list(target_dir.rglob("tasks.jsonl"))
    files_to_process.extend([(f, 'tasks') for f in tasks_files if BACKUP_SUFFIX not in f.suffixes])
    print(f"找到 {len(tasks_files)} 个 tasks.jsonl 文件")

print(f"\n总共需要处理 {len(files_to_process)} 个文件")

if len(files_to_process) == 0:
    print(f"\n没有选择任何处理类型或未找到对应文件")
else:
    # 处理所有文件
    print("\n" + "=" * 100)
    print("开始处理文件...")
    print("=" * 100)
    
    results = []
    for i, (file_path, process_type) in enumerate(files_to_process, 1):
        print(f"\n进度: [{i}/{len(files_to_process)}]")
        result = process_jsonl_file(
            file_path,
            process_type,
            create_backup=CREATE_BACKUP,
            overwrite=OVERWRITE_ORIGINAL,
            output_dir=OUTPUT_DIR
        )
        results.append(result)
    
    # 显示处理结果汇总
    print("\n" + "=" * 100)
    print("处理结果汇总")
    print("=" * 100)
    
    success_count = sum(1 for r in results if r['success'])
    modified_count = sum(1 for r in results if r['modified'])
    error_count = len(results) - success_count
    
    print(f"\n总文件数: {len(results)}")
    print(f"成功处理: {success_count}")
    print(f"已修改: {modified_count}")
    print(f"失败: {error_count}")
    
    # 显示详细结果
    print(f"\n详细结果:")
    for r in results:
        status = "✓" if r['success'] else "✗"
        modified = " [已修改]" if r.get('modified') else ""
        print(f"  {status} {r['name']}{modified}")
        if r.get('error'):
            print(f"      错误: {r['error']}")
    
    # 保存处理日志
    if SAVE_LOG:
        log_lines = []
        log_lines.append("=" * 100)
        log_lines.append(f"JSONL文件处理日志")
        log_lines.append(f"处理时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        log_lines.append("=" * 100)
        log_lines.append(f"\n目标目录: {DATA_DIR}")
        log_lines.append(f"\n处理配置:")
        log_lines.append(f"  - 处理 episodes.jsonl: {PROCESS_EPISODES}")
        log_lines.append(f"  - 处理 episodes_stats.jsonl: {PROCESS_EPISODES_STATS}")
        log_lines.append(f"  - 处理 tasks.jsonl: {PROCESS_TASKS}")
        log_lines.append(f"  - 创建备份: {CREATE_BACKUP}")
        log_lines.append(f"  - 覆盖原文件: {OVERWRITE_ORIGINAL}")
        log_lines.append(f"\n处理结果:")
        log_lines.append(f"  - 总文件数: {len(results)}")
        log_lines.append(f"  - 成功处理: {success_count}")
        log_lines.append(f"  - 已修改: {modified_count}")
        log_lines.append(f"  - 失败: {error_count}")
        
        if error_count > 0:
            log_lines.append(f"\n失败文件详情:")
            for r in results:
                if not r['success']:
                    log_lines.append(f"  - {r['name']}: {r.get('error', 'Unknown error')}")
        
        log_lines.append(f"\n处理文件列表:")
        for r in results:
            status = "✓" if r['success'] else "✗"
            modified = " [已修改]" if r.get('modified') else ""
            log_lines.append(f"  {status} {r['name']}{modified}")
            if r.get('modified'):
                log_lines.append(f"      行数: {r.get('line_count', 0)}")
        
        # 保存日志文件
        log_dir = Path(os.getcwd())
        log_filename = f"jsonl_processing_log_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"
        log_filepath = log_dir / log_filename
        
        with open(log_filepath, 'w', encoding='utf-8') as f:
            f.write('\n'.join(log_lines))
        
        print(f"\n处理日志已保存至: {log_filepath}")
    
    print("\n" + "=" * 100)
    print(f"处理完成时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 100)

JSONL文件处理 - episodes_stats统计量数组拼接
开始时间: 2026-05-12 17:59:07

目标目录: /data/SJJ/UBT/lerobot_0.5.1/datasets/Part_Sorting/26_5_11_episode_1000_obj_4/meta
目录是否存在: True

处理配置:
  - 处理 episodes.jsonl: True
  - 处理 episodes_stats.jsonl: True
  - 处理 tasks.jsonl: True
  - 创建备份: False
  - 覆盖原文件: True
  - 备份后缀: _backup

找到 1 个 episodes.jsonl 文件
找到 1 个 episodes_stats.jsonl 文件
找到 1 个 tasks.jsonl 文件

总共需要处理 3 个文件

开始处理文件...

进度: [1/3]

处理文件: episodes.jsonl
  原始行数: 756
  处理类型: episodes文件
      修改tasks: 1个任务 -> Part Sorting
      修改tasks: 1个任务 -> Part Sorting
      修改tasks: 1个任务 -> Part Sorting
      修改tasks: 1个任务 -> Part Sorting
      修改tasks: 1个任务 -> Part Sorting
      修改tasks: 1个任务 -> Part Sorting
      修改tasks: 1个任务 -> Part Sorting
      修改tasks: 1个任务 -> Part Sorting
      修改tasks: 1个任务 -> Part Sorting
      修改tasks: 1个任务 -> Part Sorting
      修改tasks: 1个任务 -> Part Sorting
      修改tasks: 1个任务 -> Part Sorting
      修改tasks: 1个任务 -> Part Sorting
      修改tasks: 1个任务 -> Part Sorting
      修改tasks: 1个任务 ->

## 处理 info json文件

In [4]:
# Jupyter Notebook 单元格代码
import os
from pathlib import Path
import json
from datetime import datetime

PROJECT_ROOT = Path.cwd().parent.resolve()

# ========== 全局变量定义 ==========
# 数据目录路径（包含info.json文件的目录）
DATA_DIR = str(PROJECT_ROOT / "datasets/Part_Sorting/26_5_11_episode_1000_obj_4")

# 文件名（通常是info.json或info.jsonl）
INFO_FILE_NAME = "info.json"

# 是否创建备份（True: 备份原始文件, False: 不备份）
CREATE_BACKUP = False

# 备份文件后缀
BACKUP_SUFFIX = "_backup"

# 是否覆盖原文件（True: 覆盖原文件, False: 保存到新文件）
OVERWRITE_ORIGINAL = True

# 输出目录（当OVERWRITE_ORIGINAL=False时使用）
OUTPUT_DIR = None

# 是否保存处理日志
SAVE_LOG = False
# =================================================

def process_info_json(data, file_path):
    """
    处理info.json文件
    删除observation.environment_state，将其names拼接到observation.state的names中
    并更新observation.state的shape维度
    """
    modified = False
    
    # 检查是否存在features字段
    if 'features' not in data:
        print("  错误: 文件中没有 'features' 字段")
        return modified
    
    features = data['features']
    
    # 检查是否存在需要处理的字段
    if 'observation.state' not in features:
        print("  错误: features中没有 'observation.state' 字段")
        return modified
    
    if 'observation.environment_state' not in features:
        print("  无需处理: features中没有 'observation.environment_state' 字段")
        return modified
    
    state_info = features['observation.state']
    env_state_info = features['observation.environment_state']
    
    print(f"  处理前:")
    print(f"    observation.state.shape: {state_info.get('shape')}")
    print(f"    observation.state.names长度: {len(state_info.get('names', []))}")
    print(f"    observation.environment_state.shape: {env_state_info.get('shape')}")
    print(f"    observation.environment_state.names长度: {len(env_state_info.get('names', []))}")
    
    # 拼接names
    if 'names' in state_info and 'names' in env_state_info:
        original_names = state_info['names']
        env_names = env_state_info['names']
        
        if original_names and env_names:
            # 拼接names
            new_names = original_names + env_names
            state_info['names'] = new_names
            print(f"  拼接names: {len(original_names)} + {len(env_names)} = {len(new_names)}")
            modified = True
        else:
            print(f"  警告: names字段为空或不存在")
    else:
        print(f"  警告: names字段不存在")
    
    # 更新shape维度
    if 'shape' in state_info and 'shape' in env_state_info:
        original_shape = state_info['shape']
        env_shape = env_state_info['shape']
        
        # 检查shape是否是列表且长度相同（都是1维数组）
        if isinstance(original_shape, list) and isinstance(env_shape, list):
            if len(original_shape) == 1 and len(env_shape) == 1:
                new_shape = [original_shape[0] + env_shape[0]]
                state_info['shape'] = new_shape
                print(f"  更新shape: {original_shape} + {env_shape} = {new_shape}")
                modified = True
            else:
                # 对于多维shape，只调整第一维
                new_shape = original_shape.copy()
                new_shape[0] = original_shape[0] + env_shape[0]
                state_info['shape'] = new_shape
                print(f"  更新shape: {original_shape} + {env_shape} = {new_shape}")
                modified = True
        else:
            print(f"  警告: shape格式不正确")
    
    # 删除observation.environment_state
    del features['observation.environment_state']
    print(f"  已删除 observation.environment_state")
    modified = True
    
    return modified

def process_json_file(file_path, create_backup=True, overwrite=True, output_dir=None):
    """
    处理单个json文件
    """
    file_info = {
        'path': file_path,
        'name': file_path.name,
        'size_mb': round(file_path.stat().st_size / 1024 / 1024, 2),
        'success': False,
        'modified': False,
        'error': None
    }
    
    try:
        print(f"\n处理文件: {file_path.name}")
        
        # 读取JSON文件
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        print(f"  原始文件大小: {file_info['size_mb']} MB")
        
        # 处理数据
        modified = process_info_json(data, file_path)
        file_info['modified'] = modified
        
        if not modified:
            print(f"  无需修改")
            file_info['success'] = True
            return file_info
        
        # 保存修改后的数据
        if overwrite:
            # 如果需要备份，先创建备份
            if create_backup:
                backup_path = file_path.with_suffix(f'.json{BACKUP_SUFFIX}')
                if not backup_path.exists():
                    # 备份原始内容
                    with open(file_path, 'r', encoding='utf-8') as f:
                        original_content = f.read()
                    with open(backup_path, 'w', encoding='utf-8') as f:
                        f.write(original_content)
                    print(f"  已创建备份: {backup_path.name}")
                else:
                    print(f"  备份文件已存在: {backup_path.name}")
            
            # 覆盖原文件
            with open(file_path, 'w', encoding='utf-8') as f:
                json.dump(data, f, indent=4, ensure_ascii=False)
            print(f"  已覆盖原文件: {file_path.name}")
        else:
            # 保存到新文件
            if output_dir is None:
                output_path = file_path.parent / f"{file_path.stem}_processed.json"
            else:
                output_path = Path(output_dir) / f"{file_path.stem}_processed.json"
                output_path.parent.mkdir(parents=True, exist_ok=True)
            
            with open(output_path, 'w', encoding='utf-8') as f:
                json.dump(data, f, indent=4, ensure_ascii=False)
            print(f"  已保存到新文件: {output_path}")
        
        # 显示处理后的信息
        if 'features' in data:
            if 'observation.state' in data['features']:
                state_info = data['features']['observation.state']
                print(f"  处理后observation.state信息:")
                print(f"    shape: {state_info.get('shape')}")
                print(f"    names数量: {len(state_info.get('names', []))}")
        
        file_info['success'] = True
        
    except Exception as e:
        file_info['error'] = str(e)
        print(f"  错误: {e}")
    
    return file_info

# 开始处理
print("=" * 100)
print("Info.json文件处理 - 删除observation.environment_state并拼接names")
print(f"开始时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 100)

# 检查目录
target_dir = Path(DATA_DIR)
print(f"\n目标目录: {target_dir}")
print(f"目录是否存在: {target_dir.exists()}")

if not target_dir.exists():
    raise FileNotFoundError(f"目录不存在: {target_dir}")

# 查找info.json文件
info_files = list(target_dir.rglob(INFO_FILE_NAME))

print(f"\n找到 {len(info_files)} 个 {INFO_FILE_NAME} 文件")

if len(info_files) == 0:
    print(f"目录中没有找到任何 {INFO_FILE_NAME} 文件")
else:
    # 显示处理配置
    print(f"\n处理配置:")
    print(f"  - 创建备份: {CREATE_BACKUP}")
    print(f"  - 覆盖原文件: {OVERWRITE_ORIGINAL}")
    print(f"  - 备份后缀: {BACKUP_SUFFIX}")
    if not OVERWRITE_ORIGINAL:
        print(f"  - 输出目录: {OUTPUT_DIR if OUTPUT_DIR else '原文件目录'}")
    
    # 处理所有文件
    print("\n" + "=" * 100)
    print("开始处理文件...")
    print("=" * 100)
    
    results = []
    for i, file_path in enumerate(info_files, 1):
        print(f"\n进度: [{i}/{len(info_files)}]")
        result = process_json_file(
            file_path,
            create_backup=CREATE_BACKUP,
            overwrite=OVERWRITE_ORIGINAL,
            output_dir=OUTPUT_DIR
        )
        results.append(result)
    
    # 显示处理结果汇总
    print("\n" + "=" * 100)
    print("处理结果汇总")
    print("=" * 100)
    
    success_count = sum(1 for r in results if r['success'])
    modified_count = sum(1 for r in results if r['modified'])
    error_count = len(results) - success_count
    
    print(f"\n总文件数: {len(results)}")
    print(f"成功处理: {success_count}")
    print(f"已修改: {modified_count}")
    print(f"失败: {error_count}")
    
    # 显示详细结果
    print(f"\n详细结果:")
    for r in results:
        status = "✓" if r['success'] else "✗"
        modified = " [已修改]" if r.get('modified') else ""
        print(f"  {status} {r['name']}{modified}")
        if r.get('error'):
            print(f"      错误: {r['error']}")
    
    # 保存处理日志
    if SAVE_LOG:
        log_lines = []
        log_lines.append("=" * 100)
        log_lines.append(f"Info.json文件处理日志")
        log_lines.append(f"处理时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        log_lines.append("=" * 100)
        log_lines.append(f"\n目标目录: {DATA_DIR}")
        log_lines.append(f"\n处理配置:")
        log_lines.append(f"  - 创建备份: {CREATE_BACKUP}")
        log_lines.append(f"  - 覆盖原文件: {OVERWRITE_ORIGINAL}")
        log_lines.append(f"\n处理结果:")
        log_lines.append(f"  - 总文件数: {len(results)}")
        log_lines.append(f"  - 成功处理: {success_count}")
        log_lines.append(f"  - 已修改: {modified_count}")
        log_lines.append(f"  - 失败: {error_count}")
        
        if error_count > 0:
            log_lines.append(f"\n失败文件详情:")
            for r in results:
                if not r['success']:
                    log_lines.append(f"  - {r['name']}: {r.get('error', 'Unknown error')}")
        
        log_lines.append(f"\n处理文件列表:")
        for r in results:
            status = "✓" if r['success'] else "✗"
            modified = " [已修改]" if r.get('modified') else ""
            log_lines.append(f"  {status} {r['name']}{modified}")
        
        # 保存日志文件
        log_dir = Path(os.getcwd())
        log_filename = f"info_json_processing_log_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"
        log_filepath = log_dir / log_filename
        
        with open(log_filepath, 'w', encoding='utf-8') as f:
            f.write('\n'.join(log_lines))
        
        print(f"\n处理日志已保存至: {log_filepath}")
    
    print("\n" + "=" * 100)
    print(f"处理完成时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 100)

Info.json文件处理 - 删除observation.environment_state并拼接names
开始时间: 2026-05-12 17:59:00

目标目录: /data/SJJ/UBT/lerobot_0.5.1/datasets/Part_Sorting/26_5_11_episode_1000_obj_4
目录是否存在: True

找到 1 个 info.json 文件

处理配置:
  - 创建备份: False
  - 覆盖原文件: True
  - 备份后缀: _backup

开始处理文件...

进度: [1/1]

处理文件: info.json
  原始文件大小: 0.01 MB
  处理前:
    observation.state.shape: [20]
    observation.state.names长度: 20
    observation.environment_state.shape: [28]
    observation.environment_state.names长度: 28
  拼接names: 20 + 28 = 48
  更新shape: [20] + [28] = [48]
  已删除 observation.environment_state
  已覆盖原文件: info.json
  处理后observation.state信息:
    shape: [48]
    names数量: 48

处理结果汇总

总文件数: 1
成功处理: 1
已修改: 1
失败: 0

详细结果:
  ✓ info.json [已修改]

处理完成时间: 2026-05-12 17:59:00
